# 02 — Building a catalog

The catalog is built in two steps.

1. **Base catalog, once.** Every galaxy of the mock with a line above 1.0e-16 erg/s/cm² inside the grism bandpass,
   with its position, redshifts and the flux of all nine lines. It is written by one batch job:

   ```bash
   sbatch scripts/production.sh          # configs/production.yaml; one CPU node, about 5 minutes
   ```

2. **Catalog, as often as needed.** `base.make_catalog(...)` applies the flux limits, assigns the redshifts with a
   confusion matrix and selects the sample. It takes about 20 s on a login node.

This notebook does step 2 for the galaxies of every line at 1.5 < z < 1.9.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")      # diffsky imports JAX; keep it off the GPU

import numpy as np
import matplotlib.pyplot as plt
from romanmock import plotting

plotting.use_style()

MOCK = "sdss_feniks_hizels_260710_09_07_2026"                          # version of the diffsky mock
ROOT = f"/global/cfs/cdirs/m4943/Simulations/RomanMocks/{MOCK}"        # products of romanmock for this mock
CATALOG = f"{ROOT}/catalogs/Every_z1p5_1p9.hdf5"                       # the catalog this notebook writes
AREA_DEG2 = None                                                       # sky area [deg^2] if you know it; None: estimated

## The base catalog

In [2]:
from romanmock import BaseCatalog, Catalog

base = BaseCatalog(f"{ROOT}/base", area_deg2=AREA_DEG2)
print(base)
print(f"sky area: {base.area_deg2:.2f} deg2 ({base.area_source})")

<BaseCatalog /global/cfs/cdirs/m4943/Simulations/RomanMocks/sdss_feniks_hizels_260710_09_07_2026/base: 64,556,046 galaxies in 66 slices, 0.02 < z < 4.00; flux limit 1.00e-16 erg/s/cm2 in 10000-19300 A; lines ['Ha', 'Hb', 'OIII_5007', 'OIII_4959', 'OII_3727', 'SIII_9532', 'SIII_9069', 'OIII_4363', 'SIII_6312']>
sky area: 2062.65 deg2 (footprint)


## Sky area

`area_deg2` is the sky area of the lightcone, used for every dN/dz. Give it if you know it. With `None` it is
estimated: from the sky patches the mock is made of or, for a mock without a patch decomposition, from the
galaxies. `base.area_source` tells which: `user`, `footprint` or `healpix`.

## The catalog

The confusion matrix gives, for a galaxy with a single detected line, the probability that the line is taken
for another one. Lines that are not listed are identified correctly. A galaxy with several detected lines
always gets its correct redshift. The probabilities below are an assumed scenario, not a calibration of
the Roman pipeline: replace them with yours.

In [3]:
confusion = {                                  # {true line: {line it is taken for: probability}}
    "Ha":        {"OIII_5007": 0.3},
    "OIII_5007": {"Ha": 0.3},
    "SIII_9532": {"Ha": 0.3},
    # "Hb":        {"OIII_5007": 0.2},
}

cat = base.make_catalog(
    flux_limit=1.5e-16,               # primary limit [erg/s/cm^2]: one line of the galaxy above it
    secondary_flux_limit=8.0e-17,     # limit for the other lines of such a galaxy (None: no secondary limit)
    confusion=confusion,
    lines=None,                      # galaxies in which every line is identified (["Ha"]: only Ha line)
    z_range=(1.5, 1.9),               # range of the assigned redshift
    columns=["gal_id", "ra", "dec", "redshift_true", "redshift_rsd", "redshift_assigned",
             "id_outcome", "n_det", "line_assigned", "line_bitmask"],
    seed=2026,
    write_to=CATALOG,
)

[romanmock] 66 slices of base, 8 worker(s)
[romanmock] sky area 2062.65 deg2 (estimated from the sky patches of the mock)
[romanmock] 44,929,655 of 64,556,046 base galaxies detected; 7,931,502 in the catalog
[romanmock] lines inside 10000 - 19300 A for redshift_assigned in 1.5 - 1.9 (lightcone: z_rsd 0.01 - 4.06)
  line        rest [A]    redshift      observed [A]       detected   identified    brightest    mistaken  interlopers
  Ha            6564.7  1.500 - 1.900   16412 - 19038     7,888,827    7,410,700    7,410,700     478,127       42,675
  Hb            4862.8  1.500 - 1.900   12157 - 14102       829,538      829,538            0           0            0
  OIII_5007     5008.3  1.500 - 1.900   12521 - 14524     5,776,002    5,733,337            0      42,665      478,127
  OIII_4959     4960.4  1.500 - 1.900   12401 - 14385       293,082      293,082            0           0            0
  OII_3727      3728.6  1.682 - 1.900   10000 - 10813       175,909      175,909          

The report lists the lines inside the bandpass for the redshift cut and the line interlopers: galaxies with a
single detected line that was taken for another line, which puts them at a wrong redshift.

In [4]:
print(cat.summary())

<Catalog: 7,931,502 galaxies, 10 columns ['gal_id', 'ra', 'dec', 'redshift_true', 'redshift_rsd', 'redshift_assigned', 'id_outcome', 'n_det', 'line_assigned', 'line_bitmask']; F > 1.50e-16 (other lines > 8.00e-17); redshift_assigned in (1.5, 1.9)>
  MULTI_LINE         5,733,337  ( 72.29%)
  SINGLE_CORRECT     1,677,363  ( 21.15%)
  SINGLE_MISID         520,802  (  6.57%)


In [5]:
from astropy.table import Table

table = Table({name: cat[name][:5] for name in cat.columns})
for name in ("ra", "dec", "redshift_true", "redshift_rsd", "redshift_assigned"):
    table[name].format = ".5f"
table

gal_id,ra,dec,redshift_true,redshift_rsd,redshift_assigned,id_outcome,n_det,line_assigned,line_bitmask
int64,float64,float64,float32,float64,float64,int8,uint8,int8,uint32
316111566,89.98400,63.91182,2.80225,2.79702,1.89679,3,1,0,4
316112146,87.39946,63.55288,2.80044,2.79650,1.89640,3,1,0,4
316113333,87.84944,63.65046,2.79432,2.79433,1.89475,3,1,0,4
316114527,89.08058,64.09690,2.78174,2.78005,1.88385,3,1,0,4
316114637,89.51220,64.11320,2.79346,2.78636,1.88866,3,1,0,4


| column | content |
|---|---|
| `gal_id` | identifier of the galaxy in the diffsky mock, to join any of its other columns |
| `ra`, `dec` | position [degree] |
| `redshift_true` | cosmological redshift |
| `redshift_rsd` | redshift including the peculiar velocity |
| `redshift_assigned` | measured redshift: `redshift_rsd` when the identification is right, displaced for an interloper |
| `id_outcome` | 1 = several lines, 2 = single line identified correctly, 3 = single line misidentified (interloper) |
| `n_det` | number of detected lines |
| `line_assigned` | index of the identified line (the brightest one of a multi-line galaxy); `cat.bit("Ha")` gives the index of a line |
| `line_bitmask` | detected lines, bit `i` = line `i` of `cat.line_names` |

`base.columns` lists every column a catalog can contain, among them the nine line fluxes.

## Reading it back

In [6]:
cat = Catalog.read(CATALOG)
print(cat)
print(f"sky area: {cat.area_deg2:.2f} deg2 ({cat.info['area_source']})")
print("settings:", {k: cat.settings[k] for k in ("flux_limit", "secondary_flux_limit", "lines", "z_range", "seed")})

galaxies, interlopers = cat.subset("galaxies"), cat.subset("interlopers")
print(f"{len(galaxies):,} galaxies with a correct redshift, {len(interlopers):,} interlopers "
      f"({100 * len(interlopers) / len(cat):.2f} %)")

<Catalog: 7,931,502 galaxies, 10 columns ['gal_id', 'ra', 'dec', 'redshift_true', 'redshift_rsd', 'redshift_assigned', 'id_outcome', 'n_det', 'line_assigned', 'line_bitmask']; F > 1.50e-16 (other lines > 8.00e-17); redshift_assigned in (1.5, 1.9)>
sky area: 2062.65 deg2 (footprint)
settings: {'flux_limit': 1.5e-16, 'secondary_flux_limit': 8e-17, 'lines': None, 'z_range': [1.5, 1.9], 'seed': 2026}
7,410,700 galaxies with a correct redshift, 520,802 interlopers (6.57 %)


The file is plain HDF5, one dataset per column, and can also be read with `h5py`. The settings of the call
are stored in the file attribute `romanmock_catalog`.

The sky area can also be given per call, `base.make_catalog(..., area_deg2=2062.65)`. It sets `cat.area_deg2`,
the normalisation of `cat.dndz(z_edges)` and of the downsampling to a reference dN/dz. It does not change which
galaxies are selected.

## Other samples

Every argument of `make_catalog` can be changed freely between calls; `help(base.make_catalog)` describes
them all. Two examples:

```python
# [O III] at 2.0 < z < 2.8, on a part of the sky, with the line fluxes
base.make_catalog(lines=["OIII_5007"], z_range=(2.0, 2.8), cuts={"dec": (60.0, 70.0)},
                  columns=["ra", "dec", "redshift_assigned", "id_outcome", "OIII_5007_flux", "Hb_flux"])

# the Halpha sample thinned to the number density forecast by Wang et al. (2022)
base.make_catalog(lines=["Ha"], z_range=(1.0, 1.9), downsample="wang2022")
```

From the command line:

```bash
romanmock catalog $ROOT/base -o Every_z1p5_1p9.hdf5 --flux-limit 1.5e-16 --secondary-flux-limit 8.0e-17 \
          --confusion confusion.yaml --z-range 1.5 1.9 --seed 2026 --area 2062.65
```

In a script, call `make_catalog` below `if __name__ == "__main__":`, because the slices of the base catalog
are read by worker processes.